In [3]:
!pip install zstandard

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 166.4 MB/s eta 0:00:00


In [ ]:
pip show zstandard

In [4]:
!mkdir -p ./data
from transformers import AutoTokenizer
from datasets import load_dataset, Dataset
from tqdm import tqdm
import time
import logging
logging.basicConfig(level=logging.WARNING)

# Function to load and collect a fixed number of examples with retries
def get_slimpajama_data(num_train=10000, num_eval=1000, max_retries=5, delay=2):
    def collect_examples(stream, num_examples, split_name):
        examples = []
        iterator = iter(stream)
        pbar = tqdm(total=num_examples, desc=f"Collecting {split_name} examples")
        while len(examples) < num_examples:
            try:
                example = next(iterator)
                examples.append(example)
                pbar.update(1)
            except StopIteration:
                logging.warning(f"Reached end of stream before collecting {num_examples} examples.")
                break
            except Exception as e:
                logging.warning(f"Error while collecting {split_name} example {len(examples)}: {e}")
                retries = 0
                while retries < max_retries:
                    try:
                        time.sleep(delay)
                        example = next(iterator)
                        examples.append(example)
                        pbar.update(1)
                        break
                    except Exception as e_retry:
                        retries += 1
                        logging.warning(f"Retry {retries}/{max_retries} failed: {e_retry}")
                else:
                    logging.warning(f"Skipping {split_name} example after {max_retries} retries.")
        pbar.close()
        return examples

    print(f"Loading {num_train} training examples and {num_eval} evaluation examples...")

    train_stream = load_dataset("cerebras/SlimPajama-627B", split="train", streaming=True)
    eval_stream = load_dataset("cerebras/SlimPajama-627B", split="validation", streaming=True)

    train_examples = collect_examples(train_stream, num_train, "training")
    eval_examples = collect_examples(eval_stream, num_eval, "evaluation")

    train_dataset = Dataset.from_dict({
        'text': [example['text'] for example in train_examples]
    })

    eval_dataset = Dataset.from_dict({
        'text': [example['text'] for example in eval_examples]
    })

    return train_dataset, eval_dataset

# Load and preprocess the data
print("Loading train and evaluation datasets...")
train_dataset, eval_dataset = get_slimpajama_data(num_train=10000, num_eval=1000)


Loading train and evaluation datasets...
Loading 10000 training examples and 1000 evaluation examples...


In [ ]:
# Load tokenizer (you can replace "gpt2-xl" with another model if needed)
tokenizer = AutoTokenizer.from_pretrained("gpt2-large")
tokenizer.pad_token = tokenizer.eos_token  # GPT-2 doesn't have pad_token by default

# Tokenize dataset for training
def preprocess_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=512,
        return_tensors=None,
    )

# Tokenize datasets
print("Tokenizing train dataset...")
tokenized_train_dataset = train_dataset.map(
    preprocess_function,
    batched=True,
    batch_size=16,
    remove_columns=["text"]
)

print("Tokenizing evaluation dataset...")
tokenized_eval_dataset = eval_dataset.map(
    preprocess_function,
    batched=True,
    batch_size=16,
    remove_columns=["text"]
)

# Add labels for causal language modeling
tokenized_train_dataset = tokenized_train_dataset.map(
    lambda examples: {"labels": examples["input_ids"]},
    batched=True
)

tokenized_eval_dataset = tokenized_eval_dataset.map(
    lambda examples: {"labels": examples["input_ids"]},
    batched=True
)

print("Tokenization complete.")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from tqdm import tqdm
import math

device = "cuda" if torch.cuda.is_available() else "cpu"

def compute_perplexity(model_id, dataset, max_length=512):
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id).to(device)
    model.eval()

    total_loss = 0.0
    total_tokens = 0

    for example in tqdm(dataset, desc=f"Evaluating {model_id}"):
        inputs = tokenizer(example['text'], return_tensors='pt', truncation=True,
                           max_length=max_length).to(device)

        with torch.no_grad():
            outputs = model(**inputs, labels=inputs['input_ids'])
            loss = outputs.loss
            n_tokens = inputs['input_ids'].shape[1]
            total_loss += loss.item() * n_tokens
            total_tokens += n_tokens

    avg_neg_log_likelihood = total_loss / total_tokens
    perplexity = math.exp(avg_neg_log_likelihood)
    return perplexity

# Run perplexity evaluation
ppl_topk = compute_perplexity("stevensu123/gpt2-moe-topk", eval_dataset)
ppl_topp = compute_perplexity("stevensu123/gpt2-moe-topp", eval_dataset)

print(f"Top-K Model Perplexity: {ppl_topk:.2f}")
print(f"Top-P Model Perplexity: {ppl_topp:.2f}")


/home/ec2-user/dynamic_moe/.venv/lib64/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'Baseline'